---
title: "Implementación — Parte II"
subtitle: "Ruta Más Corta (SPP) y Agente Viajero (TSP)"
description: "Notebook con Dijkstra, formulación LP/ILP (PuLP), heurísticas TSP y análisis de escalabilidad con visualizaciones Plotly."
categories: [Parte II, Notebook, SPP, TSP, Python]
image: images/parte2_thumb.svg
format:
  html:
    code-fold: true
    code-tools: true
    toc: true
    toc-depth: 3
---

## Implementación computacional — Parte II

Este notebook presenta la implementación de los dos problemas clásicos de optimización combinatoria de la Parte II:

1. **Problema de la Ruta Más Corta (SPP)** — Algoritmo de Dijkstra desde cero + verificación LP con PuLP.
2. **Problema del Agente Viajero (TSP)** — Formulación ILP (Miller-Tucker-Zemlin), enumeración exhaustiva y heurísticas (vecino más cercano, 2-opt).

Se incluyen visualizaciones interactivas con Plotly y análisis de escalabilidad.

In [1]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import heapq
import itertools
import time
import pandas as pd
from pulp import (
    LpProblem, LpMinimize, LpVariable, LpBinary,
    LpContinuous, lpSum, value, LpStatus
)

## Funciones auxiliares de visualización

Funciones para representar grafos interactivos con Plotly.

In [2]:
def plot_graph(G, pos, title='Grafo', path=None, path_color='red',
               node_color='lightblue', node_labels=None,
               edge_labels=True, highlight_nodes=None,
               width=700, height=500):
    """
    Visualiza un grafo con Plotly de forma interactiva.
    """
    edge_traces = []
    annotations = []

    path_edges = set()
    if path:
        for i in range(len(path) - 1):
            path_edges.add((path[i], path[i+1]))

    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        is_path = (u, v) in path_edges or (v, u) in path_edges
        color = path_color if is_path else '#888'
        w = 4 if is_path else 1.5

        edge_traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode='lines',
            line=dict(width=w, color=color),
            hoverinfo='none', showlegend=False
        ))

        if edge_labels:
            weight = data.get('weight', '')
            mx, my = (x0 + x1) / 2, (y0 + y1) / 2
            annotations.append(dict(
                x=mx, y=my, text=str(weight),
                showarrow=False, font=dict(size=11, color='darkblue'),
                bgcolor='white', borderpad=2
            ))

    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]

    colors = []
    for n in G.nodes():
        if highlight_nodes and n in highlight_nodes:
            colors.append(highlight_nodes[n])
        elif path and n in path:
            colors.append(path_color)
        else:
            colors.append(node_color)

    labels = []
    for n in G.nodes():
        lbl = node_labels[n] if node_labels and n in node_labels else str(n)
        labels.append(lbl)

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        text=labels, textposition='top center',
        textfont=dict(size=14, color='black'),
        marker=dict(size=30, color=colors, line=dict(width=2, color='black')),
        hoverinfo='text',
        hovertext=[f'Nodo {n}' for n in G.nodes()],
        showlegend=False
    )

    fig = go.Figure(data=edge_traces + [node_trace])
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=16)),
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        annotations=annotations,
        width=width, height=height,
        margin=dict(l=20, r=20, t=50, b=20),
        plot_bgcolor='white'
    )
    return fig

---

# Problema de la Ruta Más Corta (SPP)

## Definición

Dado un grafo dirigido $G = (V, A)$ con costos no negativos $c_{ij} \geq 0$ en cada arco, un nodo origen $s$ y un nodo destino $t$, el problema busca el camino de $s$ a $t$ de costo mínimo.

## Formulación LP

$$\min \sum_{(i,j) \in A} c_{ij} \, x_{ij}$$

sujeto a:

$$\sum_{j:(i,j)\in A} x_{ij} - \sum_{j:(j,i)\in A} x_{ji} = \begin{cases} 1 & \text{si } i = s \\ -1 & \text{si } i = t \\ 0 & \text{en otro caso} \end{cases} \quad \forall i \in V$$

$$x_{ij} \geq 0 \quad \forall (i,j) \in A$$

**Propiedad clave:** La matriz de incidencia del grafo dirigido es **totalmente unimodular**, por lo que la relajación LP produce soluciones enteras sin imponer $x_{ij} \in \{0, 1\}$.

## Implementación del Algoritmo de Dijkstra

In [3]:
def dijkstra(graph, source):
    """Dijkstra con cola de prioridad (min-heap)."""
    dist = {node: float('inf') for node in graph}
    prev = {node: None for node in graph}
    dist[source] = 0
    visited = set()
    heap = [(0, source)]
    steps = []

    while heap:
        d_u, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)

        step = {
            'node': u, 'dist': d_u,
            'visited': set(visited),
            'distances': dict(dist),
            'relaxations': []
        }

        for v, w in graph[u]:
            if v not in visited and d_u + w < dist[v]:
                old_dist = dist[v]
                dist[v] = d_u + w
                prev[v] = u
                heapq.heappush(heap, (dist[v], v))
                step['relaxations'].append((v, old_dist, dist[v]))

        step['distances_after'] = dict(dist)
        steps.append(step)

    return dist, prev, steps


def reconstruct_path(prev, source, target):
    """Reconstruye el camino desde source a target."""
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()
    if path[0] != source:
        return []
    return path

## Ejemplo numérico: Grafo de 6 nodos

Creamos el grafo del ejemplo del informe (Figura 1) y ejecutamos Dijkstra desde el nodo 1 al nodo 6.

In [4]:
graph_adj = {
    1: [(2, 7), (4, 9), (5, 14)],
    2: [(3, 10), (5, 15)],
    3: [(6, 11)],
    4: [(2, 6), (5, 2)],
    5: [(6, 9)],
    6: []
}

G_spp = nx.DiGraph()
edges_spp = [
    (1, 2, 7), (1, 4, 9), (1, 5, 14),
    (2, 3, 10), (2, 5, 15),
    (3, 6, 11),
    (4, 2, 6), (4, 5, 2),
    (5, 6, 9)
]
for u, v, w in edges_spp:
    G_spp.add_edge(u, v, weight=w)

pos_spp = {
    1: (0, 1), 2: (1, 1.5), 3: (2, 1.5),
    4: (0.5, 0), 5: (1.5, 0), 6: (2.5, 0.5)
}

fig_spp = plot_graph(G_spp, pos_spp, title='Grafo Dirigido — Problema de Ruta Más Corta')
fig_spp.show()

In [5]:
source, target = 1, 6
dist, prev, steps = dijkstra(graph_adj, source)
shortest_path = reconstruct_path(prev, source, target)

print('=== Algoritmo de Dijkstra ===')
print(f'Origen: {source}, Destino: {target}')
print(f'Ruta más corta: {" → ".join(map(str, shortest_path))}')
print(f'Costo total: {dist[target]}')
print()
print('Distancias mínimas desde nodo 1:')
for node in sorted(dist.keys()):
    print(f'  d[{node}] = {dist[node]}')
print()
print('Pasos del algoritmo:')
for i, step in enumerate(steps):
    relax = ', '.join([f'd[{v}]: {old}→{new}' for v, old, new in step['relaxations']])
    relax = relax if relax else 'Sin relajaciones'
    print(f'  Paso {i+1}: Extraer nodo {step["node"]} (d={step["dist"]}) — {relax}')

=== Algoritmo de Dijkstra ===
Origen: 1, Destino: 6
Ruta más corta: 1 → 4 → 5 → 6
Costo total: 20

Distancias mínimas desde nodo 1:
  d[1] = 0
  d[2] = 7
  d[3] = 17
  d[4] = 9
  d[5] = 11
  d[6] = 20

Pasos del algoritmo:
  Paso 1: Extraer nodo 1 (d=0) — d[2]: inf→7, d[4]: inf→9, d[5]: inf→14
  Paso 2: Extraer nodo 2 (d=7) — d[3]: inf→17
  Paso 3: Extraer nodo 4 (d=9) — d[5]: 14→11
  Paso 4: Extraer nodo 5 (d=11) — d[6]: inf→20
  Paso 5: Extraer nodo 3 (d=17) — Sin relajaciones
  Paso 6: Extraer nodo 6 (d=20) — Sin relajaciones


In [6]:
fig_path = plot_graph(
    G_spp, pos_spp,
    title=f'Ruta Más Corta: {" → ".join(map(str, shortest_path))} (costo = {dist[target]})',
    path=shortest_path,
    path_color='#e74c3c'
)
fig_path.show()

## Visualización paso a paso de Dijkstra

In [7]:
for i, step in enumerate(steps):
    visited = step['visited']
    current = step['node']
    dists = step['distances_after']

    highlight = {}
    for n in G_spp.nodes():
        if n == current:
            highlight[n] = '#e74c3c'
        elif n in visited:
            highlight[n] = '#2ecc71'
        else:
            highlight[n] = 'lightblue'

    labels = {}
    for n in G_spp.nodes():
        d = dists.get(n, float('inf'))
        d_str = str(d) if d != float('inf') else '∞'
        labels[n] = f'{n} (d={d_str})'

    fig_step = plot_graph(
        G_spp, pos_spp,
        title=f'Paso {i+1}: Extraer nodo {current} (d={step["dist"]})',
        highlight_nodes=highlight, node_labels=labels
    )

    relaxations = step['relaxations']
    relax_text = ', '.join(
        [f'd[{v}]: {old}→{new}' for v, old, new in relaxations]
    ) if relaxations else 'Sin relajaciones'

    fig_step.add_annotation(
        x=0.5, y=-0.05, xref='paper', yref='paper',
        text=f'Relajaciones: {relax_text}',
        showarrow=False, font=dict(size=12),
        bgcolor='lightyellow', borderpad=4
    )
    fig_step.show()

## Verificación LP con PuLP

Resolvemos el mismo problema mediante programación lineal para verificar la equivalencia con Dijkstra.

In [8]:
nodes_all = list(graph_adj.keys())
arcs = [(u, v, w) for u in graph_adj for v, w in graph_adj[u]]

prob_spp = LpProblem('Ruta_Mas_Corta', LpMinimize)
x = {}
for u, v, w in arcs:
    x[(u, v)] = LpVariable(f'x_{u}_{v}', lowBound=0, cat=LpContinuous)

prob_spp += lpSum(w * x[(u, v)] for u, v, w in arcs)

s, t = 1, 6
for i in nodes_all:
    outflow = lpSum(x[(u, v)] for u, v, _ in arcs if u == i)
    inflow = lpSum(x[(u, v)] for u, v, _ in arcs if v == i)
    if i == s:
        prob_spp += outflow - inflow == 1
    elif i == t:
        prob_spp += outflow - inflow == -1
    else:
        prob_spp += outflow - inflow == 0

prob_spp.solve()

print('=== Solución LP (PuLP) ===')
print(f'Estado: {LpStatus[prob_spp.status]}')
print(f'Costo óptimo: {value(prob_spp.objective)}')
print()
print('Arcos activos (x > 0):')
for (u, v), var in x.items():
    if var.varValue and var.varValue > 0.5:
        print(f'  ({u} → {v}): x = {var.varValue}')
print()
print(f'Verificación: LP costo = {value(prob_spp.objective):.0f} '
      f'coincide con Dijkstra costo = {dist[target]}')

=== Solución LP (PuLP) ===
Estado: Optimal
Costo óptimo: 20.0

Arcos activos (x > 0):
  (1 → 4): x = 1.0
  (4 → 5): x = 1.0
  (5 → 6): x = 1.0

Verificación: LP costo = 20 coincide con Dijkstra costo = 20


## Escalabilidad de Dijkstra

Generamos grafos aleatorios de diferentes tamaños para medir el rendimiento.

In [9]:
def generate_random_graph(n_nodes, density=0.3, max_weight=20, seed=42):
    rng = np.random.default_rng(seed)
    G = nx.DiGraph()
    G.add_nodes_from(range(n_nodes))
    for i in range(n_nodes - 1):
        w = int(rng.integers(1, max_weight))
        G.add_edge(i, i + 1, weight=w)
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j and not G.has_edge(i, j) and rng.random() < density:
                w = int(rng.integers(1, max_weight))
                G.add_edge(i, j, weight=w)
    return G

sizes = [10, 50, 100, 200, 500, 1000, 2000]
times_dijkstra = []

for n in sizes:
    G_rand = generate_random_graph(n, density=0.1)
    adj = {i: [] for i in range(n)}
    for u, v, data in G_rand.edges(data=True):
        adj[u].append((v, data['weight']))
    start = time.perf_counter()
    for _ in range(5):
        dijkstra(adj, 0)
    elapsed = (time.perf_counter() - start) / 5
    times_dijkstra.append(elapsed)
    print(f'n={n:5d}: {elapsed*1000:.2f} ms')

fig_scale = go.Figure()
fig_scale.add_trace(go.Scatter(
    x=sizes, y=[t * 1000 for t in times_dijkstra],
    mode='lines+markers', name='Dijkstra',
    marker=dict(size=10), line=dict(width=2, color='#3498db')
))
fig_scale.update_layout(
    title='Escalabilidad del Algoritmo de Dijkstra',
    xaxis_title='Número de nodos', yaxis_title='Tiempo (ms)',
    xaxis_type='log', yaxis_type='log',
    width=700, height=450
)
fig_scale.show()

n=   10: 0.03 ms
n=   50: 0.13 ms
n=  100: 0.62 ms
n=  200: 2.95 ms
n=  500: 17.16 ms
n= 1000: 65.93 ms
n= 2000: 306.21 ms


---

# Problema del Agente Viajero (TSP)

## Definición

Dado un grafo completo $G = (V, E)$ con $n$ nodos y costos $c_{ij}$, el TSP busca un **circuito hamiltoniano** de costo mínimo.

## Formulación ILP (Miller-Tucker-Zemlin)

$$\min \sum_{i=1}^{n} \sum_{\substack{j=1 \\ j \neq i}}^{n} c_{ij} \, x_{ij}$$

sujeto a:

$$\sum_{\substack{j=1 \\ j \neq i}}^{n} x_{ij} = 1 \quad \forall i \qquad \sum_{\substack{i=1 \\ i \neq j}}^{n} x_{ij} = 1 \quad \forall j$$

$$u_i - u_j + n \cdot x_{ij} \leq n - 1 \quad \forall i, j = 2, \ldots, n,\; i \neq j$$

$$1 \leq u_i \leq n - 1 \quad \forall i = 2, \ldots, n \qquad x_{ij} \in \{0, 1\}$$

**A diferencia del SPP**, aquí las variables binarias son necesarias porque la matriz no es totalmente unimodular.

## Ejemplo: TSP de 5 ciudades

In [10]:
cities = ['A', 'B', 'C', 'D', 'E']
n_cities = len(cities)
dist_matrix = np.array([
    [ 0, 10, 15, 20,  6],
    [10,  0,  8, 18, 12],
    [15,  8,  0,  7, 14],
    [20, 18,  7,  0, 12],
    [ 6, 12, 14, 12,  0]
])

df_dist = pd.DataFrame(dist_matrix, index=cities, columns=cities)
print('Matriz de distancias:')
display(df_dist)

fig_heat = px.imshow(
    dist_matrix, x=cities, y=cities,
    text_auto=True, color_continuous_scale='Blues',
    title='Matriz de Distancias entre Ciudades'
)
fig_heat.update_layout(width=500, height=450)
fig_heat.show()

Matriz de distancias:


,A,B,C,D,E
A,0,10,15,20,6
B,10,0,8,18,12
C,15,8,0,7,14
D,20,18,7,0,12
E,6,12,14,12,0


## Solución exacta — Formulación MTZ con PuLP

In [11]:
def solve_tsp_mtz(dist_matrix, city_names=None):
    n = len(dist_matrix)
    if city_names is None:
        city_names = list(range(n))

    prob = LpProblem('TSP_MTZ', LpMinimize)
    x = {}
    for i in range(n):
        for j in range(n):
            if i != j:
                x[(i, j)] = LpVariable(f'x_{i}_{j}', cat=LpBinary)

    u = {}
    for i in range(1, n):
        u[i] = LpVariable(f'u_{i}', lowBound=1, upBound=n-1, cat=LpContinuous)

    prob += lpSum(dist_matrix[i][j] * x[(i, j)]
                  for i in range(n) for j in range(n) if i != j)

    for i in range(n):
        prob += lpSum(x[(i, j)] for j in range(n) if j != i) == 1
        prob += lpSum(x[(j, i)] for j in range(n) if j != i) == 1

    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                prob += u[i] - u[j] + n * x[(i, j)] <= n - 1

    prob.solve()

    if prob.status != 1:
        return None, None, LpStatus[prob.status]

    tour = [0]
    current = 0
    for _ in range(n - 1):
        for j in range(n):
            if j != current and (current, j) in x and x[(current, j)].varValue > 0.5:
                tour.append(j)
                current = j
                break
    tour.append(0)

    cost = value(prob.objective)
    return tour, cost, LpStatus[prob.status]


tour_opt, cost_opt, status = solve_tsp_mtz(dist_matrix, cities)

print('=== Solución Exacta (MTZ con PuLP) ===')
print(f'Estado: {status}')
tour_names = [cities[i] for i in tour_opt]
print(f'Tour óptimo: {" → ".join(tour_names)}')
print(f'Costo total: {cost_opt}')

=== Solución Exacta (MTZ con PuLP) ===
Estado: Optimal
Tour óptimo: A → B → C → D → E → A
Costo total: 43.0


In [12]:
city_pos = {
    'A': (0, 1.5), 'B': (1.5, 2.5), 'C': (3, 1.5),
    'D': (2.5, 0), 'E': (0.5, 0)
}

G_tsp = nx.complete_graph(cities)
for i, c1 in enumerate(cities):
    for j, c2 in enumerate(cities):
        if i < j:
            G_tsp[c1][c2]['weight'] = dist_matrix[i][j]

fig_tour = plot_graph(
    G_tsp, city_pos,
    title=f'Tour Óptimo TSP: {" → ".join(tour_names)} (costo = {int(cost_opt)})',
    path=tour_names, path_color='#e74c3c', node_color='#3498db'
)
fig_tour.show()

## Verificación por enumeración exhaustiva

Con 5 ciudades hay $(5-1)!/2 = 12$ tours distintos. Evaluamos todos para confirmar el óptimo.

In [13]:
def tour_cost(tour, dist_mat):
    return sum(dist_mat[tour[i]][tour[i+1]] for i in range(len(tour)-1))

all_tours = [[0]+list(p)+[0] for p in itertools.permutations(range(1, n_cities))]
tour_costs = sorted([(tour_cost(t, dist_matrix), t) for t in all_tours])

print(f'Total de tours: {len(all_tours)}')
print(f'Tours únicos (simétrico): {len(all_tours) // 2}\n')
print('Todos los tours ordenados por costo:')
print('-' * 55)
for cost, tour in tour_costs:
    names = ' → '.join(cities[i] for i in tour)
    marker = ' ← ÓPTIMO' if cost == tour_costs[0][0] else ''
    print(f'  {names}: costo = {cost}{marker}')

fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=[c for c, _ in tour_costs], nbinsx=10,
    marker_color='#3498db', marker_line=dict(width=1, color='black')
))
fig_hist.add_vline(x=tour_costs[0][0], line_dash='dash', line_color='red',
                   annotation_text=f'Óptimo = {tour_costs[0][0]}')
fig_hist.update_layout(
    title='Distribución de Costos de Tours',
    xaxis_title='Costo del tour', yaxis_title='Frecuencia',
    width=600, height=400
)
fig_hist.show()

Total de tours: 24
Tours únicos (simétrico): 12

Todos los tours ordenados por costo:
-------------------------------------------------------
  A → B → C → D → E → A: costo = 43 ← ÓPTIMO
  A → E → D → C → B → A: costo = 43 ← ÓPTIMO
  A → D → C → B → E → A: costo = 53
  A → E → B → C → D → A: costo = 53
  A → B → D → C → E → A: costo = 55
  A → E → C → D → B → A: costo = 55
  A → B → E → D → C → A: costo = 56
  A → C → D → E → B → A: costo = 56
  A → C → D → B → E → A: costo = 58
  A → E → B → D → C → A: costo = 58
  A → C → B → D → E → A: costo = 59
  A → E → D → B → C → A: costo = 59
  A → B → E → C → D → A: costo = 63
  A → D → C → E → B → A: costo = 63
  A → B → C → E → D → A: costo = 64
  A → D → E → C → B → A: costo = 64
  A → D → B → C → E → A: costo = 66
  A → E → C → B → D → A: costo = 66
  A → C → B → E → D → A: costo = 67
  A → D → E → B → C → A: costo = 67
  A → B → D → E → C → A: costo = 69
  A → C → E → D → B → A: costo = 69
  A → C → E → B → D → A: costo = 79
  A → D → B 

## Heurísticas: Vecino más cercano y 2-opt

In [14]:
def nearest_neighbor_tsp(dist_matrix, start=0):
    n = len(dist_matrix)
    visited = {start}; tour = [start]; current = start; total_cost = 0
    while len(visited) < n:
        nearest = min((j for j in range(n) if j not in visited),
                      key=lambda j: dist_matrix[current][j])
        total_cost += dist_matrix[current][nearest]
        visited.add(nearest); tour.append(nearest); current = nearest
    total_cost += dist_matrix[current][start]
    tour.append(start)
    return tour, total_cost


def two_opt(tour, dist_matrix):
    n = len(tour) - 1; improved = True; best = list(tour)
    history = [tour_cost(best, dist_matrix)]
    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                new = best[:i] + best[i:j+1][::-1] + best[j+1:]
                if tour_cost(new, dist_matrix) < tour_cost(best, dist_matrix):
                    best = new; improved = True
                    history.append(tour_cost(best, dist_matrix))
    return best, tour_cost(best, dist_matrix), history


tour_nn, cost_nn = nearest_neighbor_tsp(dist_matrix)
tour_2opt, cost_2opt, hist_2opt = two_opt(tour_nn, dist_matrix)

print('=== Heurísticas ===')
print(f'Vecino más cercano: {" → ".join(cities[i] for i in tour_nn)} = {cost_nn}')
print(f'Después de 2-opt:   {" → ".join(cities[i] for i in tour_2opt)} = {cost_2opt}')
print(f'Óptimo exacto:      {int(cost_opt)}')
print(f'Gap NN: {(cost_nn-cost_opt)/cost_opt*100:.1f}% | Gap 2-opt: {(cost_2opt-cost_opt)/cost_opt*100:.1f}%')

=== Heurísticas ===
Vecino más cercano: A → E → B → C → D → A = 53
Después de 2-opt:   A → E → D → C → B → A = 43
Óptimo exacto:      43
Gap NN: 23.3% | Gap 2-opt: 0.0%


## TSP con ciudades aleatorias (10 ciudades)

In [15]:
def generate_tsp_instance(n, seed=42):
    rng = np.random.default_rng(seed)
    coords = rng.random((n, 2)) * 100
    dist_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist_mat[i][j] = np.sqrt(
                (coords[i][0]-coords[j][0])**2 + (coords[i][1]-coords[j][1])**2)
    return coords, dist_mat


def plot_tsp_tour(coords, tour, title='Tour TSP', color='#e74c3c'):
    fig = go.Figure()
    x_e, y_e = [], []
    for i in range(len(tour)-1):
        x_e.extend([coords[tour[i]][0], coords[tour[i+1]][0], None])
        y_e.extend([coords[tour[i]][1], coords[tour[i+1]][1], None])
    fig.add_trace(go.Scatter(x=x_e, y=y_e, mode='lines',
                             line=dict(color=color, width=2), showlegend=False))
    fig.add_trace(go.Scatter(
        x=coords[:,0], y=coords[:,1], mode='markers+text',
        text=[str(i) for i in range(len(coords))], textposition='top center',
        marker=dict(size=12, color='#3498db', line=dict(width=1, color='black')),
        showlegend=False))
    cost = sum(np.sqrt((coords[tour[i]][0]-coords[tour[i+1]][0])**2+
                       (coords[tour[i]][1]-coords[tour[i+1]][1])**2)
              for i in range(len(tour)-1))
    fig.update_layout(title=f'{title} (costo = {cost:.1f})',
                      xaxis_title='X', yaxis_title='Y',
                      width=600, height=500, plot_bgcolor='white')
    return fig

coords_10, dist_10 = generate_tsp_instance(10, seed=123)
tour_exact, cost_exact, _ = solve_tsp_mtz(dist_10)
tour_nn10, cost_nn10 = nearest_neighbor_tsp(dist_10)
tour_2opt10, cost_2opt10, _ = two_opt(tour_nn10, dist_10)

print(f'Exacto (10 ciudades): {cost_exact:.2f}')
print(f'Vecino más cercano:   {cost_nn10:.2f} (gap: {(cost_nn10-cost_exact)/cost_exact*100:.1f}%)')
print(f'NN + 2-opt:           {cost_2opt10:.2f} (gap: {(cost_2opt10-cost_exact)/cost_exact*100:.1f}%)')

fig_exact = plot_tsp_tour(coords_10, tour_exact, 'Solución Exacta (MTZ)', '#e74c3c')
fig_exact.show()

fig_nn = plot_tsp_tour(coords_10, tour_nn10, 'Vecino Más Cercano', '#f39c12')
fig_nn.show()

Exacto (10 ciudades): 286.44
Vecino más cercano:   286.44 (gap: 0.0%)
NN + 2-opt:           286.44 (gap: 0.0%)


## Comparación de escalabilidad: Exacto vs. Heurísticas

In [16]:
tsp_sizes = [5, 8, 10, 12, 15]
times_exact = []; times_nn_l = []; times_2opt_l = []
costs_exact_l = []; costs_nn_l = []; costs_2opt_l = []

for n in tsp_sizes:
    coords, dm = generate_tsp_instance(n, seed=42)

    t0 = time.perf_counter()
    tour_e, cost_e, _ = solve_tsp_mtz(dm)
    times_exact.append(time.perf_counter() - t0)
    costs_exact_l.append(cost_e)

    t0 = time.perf_counter()
    tour_h, cost_h = nearest_neighbor_tsp(dm)
    times_nn_l.append(time.perf_counter() - t0)
    costs_nn_l.append(cost_h)

    t0 = time.perf_counter()
    tour_2, cost_2, _ = two_opt(tour_h, dm)
    times_2opt_l.append(time.perf_counter() - t0)
    costs_2opt_l.append(cost_2)

    gap_nn = (cost_h - cost_e) / cost_e * 100
    gap_2opt = (cost_2 - cost_e) / cost_e * 100
    print(f'n={n:3d} | Exacto: {cost_e:.1f} ({times_exact[-1]:.3f}s) | '
          f'NN: {cost_h:.1f} ({gap_nn:+.1f}%) | '
          f'2-opt: {cost_2:.1f} ({gap_2opt:+.1f}%)')

fig_times = go.Figure()
fig_times.add_trace(go.Scatter(x=tsp_sizes, y=times_exact, mode='lines+markers',
    name='Exacto (MTZ)', line=dict(color='#e74c3c', width=2)))
fig_times.add_trace(go.Scatter(x=tsp_sizes, y=times_nn_l, mode='lines+markers',
    name='Vecino más cercano', line=dict(color='#f39c12', width=2)))
fig_times.add_trace(go.Scatter(x=tsp_sizes, y=times_2opt_l, mode='lines+markers',
    name='NN + 2-opt', line=dict(color='#2ecc71', width=2)))
fig_times.update_layout(
    title='Tiempo de Ejecución por Método (TSP)',
    xaxis_title='Número de ciudades', yaxis_title='Tiempo (s)',
    yaxis_type='log', width=700, height=450)
fig_times.show()

gaps_nn = [(cn-ce)/ce*100 for cn,ce in zip(costs_nn_l, costs_exact_l)]
gaps_2opt = [(c2-ce)/ce*100 for c2,ce in zip(costs_2opt_l, costs_exact_l)]
fig_q = go.Figure()
fig_q.add_trace(go.Bar(x=[str(s) for s in tsp_sizes], y=gaps_nn,
    name='Vecino más cercano', marker_color='#f39c12'))
fig_q.add_trace(go.Bar(x=[str(s) for s in tsp_sizes], y=gaps_2opt,
    name='NN + 2-opt', marker_color='#2ecc71'))
fig_q.update_layout(
    title='Gap de Optimalidad (% sobre el óptimo)',
    xaxis_title='Ciudades', yaxis_title='Gap (%)',
    barmode='group', width=700, height=400)
fig_q.show()

n=  5 | Exacto: 226.9 (0.046s) | NN: 226.9 (+0.0%) | 2-opt: 226.9 (+0.0%)
n=  8 | Exacto: 240.5 (0.111s) | NN: 240.5 (+0.0%) | 2-opt: 240.5 (+0.0%)
n= 10 | Exacto: 264.5 (0.178s) | NN: 264.5 (+0.0%) | 2-opt: 264.5 (+0.0%)
n= 12 | Exacto: 298.1 (0.344s) | NN: 380.5 (+27.7%) | 2-opt: 298.1 (+0.0%)
n= 15 | Exacto: 313.0 (0.272s) | NN: 339.4 (+8.4%) | 2-opt: 313.0 (-0.0%)


---

# Comparación: SPP vs TSP

In [17]:
compare_sizes = [5, 10, 15, 20, 50, 100]
t_spp_list = []; t_tsp_list = []

for n in compare_sizes:
    G_r = generate_random_graph(n, density=0.3, seed=42)
    adj = {i: [] for i in range(n)}
    for u, v, d in G_r.edges(data=True):
        adj[u].append((v, d['weight']))
    t0 = time.perf_counter()
    dijkstra(adj, 0)
    t_spp_list.append(time.perf_counter() - t0)

    if n <= 20:
        _, dm = generate_tsp_instance(n, seed=42)
        t0 = time.perf_counter()
        solve_tsp_mtz(dm)
        t_tsp_list.append(time.perf_counter() - t0)
    else:
        t_tsp_list.append(None)

fig_compare = go.Figure()
fig_compare.add_trace(go.Scatter(
    x=compare_sizes, y=[t*1000 for t in t_spp_list],
    mode='lines+markers', name='Dijkstra (SPP)',
    line=dict(color='#3498db', width=2)))

valid_tsp = [(s, t*1000) for s, t in zip(compare_sizes, t_tsp_list) if t is not None]
fig_compare.add_trace(go.Scatter(
    x=[s for s, _ in valid_tsp], y=[t for _, t in valid_tsp],
    mode='lines+markers', name='MTZ Exacto (TSP)',
    line=dict(color='#e74c3c', width=2)))

fig_compare.update_layout(
    title='Comparación de Escalabilidad: Dijkstra (SPP) vs. MTZ (TSP)',
    xaxis_title='Nodos/Ciudades', yaxis_title='Tiempo (ms)',
    yaxis_type='log', width=700, height=450)
fig_compare.show()

## Conclusiones

| Aspecto | Ruta Más Corta (SPP) | Agente Viajero (TSP) |
|:--------|:---------------------|:---------------------|
| **Objetivo** | Camino s→t mínimo | Circuito hamiltoniano mínimo |
| **Formulación** | LP (continuo) | ILP (binario) |
| **Integralidad** | Automática (unimod. total) | Variables binarias necesarias |
| **Complejidad** | P: O((V+E) log V) | NP-difícil: O(n²·2ⁿ) |
| **Algoritmo** | Dijkstra, Bellman-Ford | Branch-and-Cut, heurísticas |
| **Escalabilidad** | Millones de nodos | Miles de nodos |

La diferencia fundamental es la **complejidad estructural**: el SPP aprovecha la subestructura óptima, mientras el TSP impone una dependencia global que impide descomposición eficiente.